In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display, clear_output
import ipywidgets as widgets
import os

# --- 1. AUTOMATIC DEPENDENCY CHECK ---
try:
    import subprocess
    subprocess.run(["ffmpeg", "-version"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
except (FileNotFoundError, subprocess.CalledProcessError):
    print("Installing system video tools (ffmpeg)... Please wait a moment.")
    os.system("apt-get install ffmpeg -y")
    clear_output()

# --- 2. DISK SCHEDULING ALGORITHMS ---
class DiskAlgorithms:
    @staticmethod
    def fcfs(reqs, head):
        path = [head] + reqs
        distance = sum(abs(path[i] - path[i-1]) for i in range(1, len(path)))
        return path, distance

    @staticmethod
    def sstf(reqs, head):
        path = [head]
        remaining = list(reqs)
        current = head
        while remaining:
            closest = min(remaining, key=lambda x: abs(x - current))
            path.append(closest)
            remaining.remove(closest)
            current = closest
        distance = sum(abs(path[i] - path[i-1]) for i in range(1, len(path)))
        return path, distance

    @staticmethod
    def scan(reqs, head, max_cyl=200, direction="up"):
        path = [head]
        left = sorted([r for r in reqs if r < head], reverse=True)
        right = sorted([r for r in reqs if r >= head])
        if direction == "up":
            path += right
            if left: path += [max_cyl - 1] + left
        else:
            path += left
            if right: path += [0] + right
        return path, sum(abs(path[i] - path[i-1]) for i in range(1, len(path)))

    @staticmethod
    def cscan(reqs, head, max_cyl=200):
        right = sorted([r for r in reqs if r >= head])
        left = sorted([r for r in reqs if r < head])
        path = [head] + right + [max_cyl - 1, 0] + left
        return path, sum(abs(path[i] - path[i-1]) for i in range(1, len(path)))

    @staticmethod
    def look(reqs, head, direction="up"):
        left = sorted([r for r in reqs if r < head], reverse=True)
        right = sorted([r for r in reqs if r >= head])
        path = [head] + (right + left if direction == "up" else left + right)
        return path, sum(abs(path[i] - path[i-1]) for i in range(1, len(path)))

    @staticmethod
    def clook(reqs, head):
        left = sorted([r for r in reqs if r < head])
        right = sorted([r for r in reqs if r >= head])
        path = [head] + right + left
        return path, sum(abs(path[i] - path[i-1]) for i in range(1, len(path)))

# --- 3. INTERFACE CREATION ---
style = {'description_width': '160px'}
layout = widgets.Layout(width='400px')

disk_size_w = widgets.IntText(value=200, description='Disk Size (Max Cylinders):', style=style, layout=layout)
start_head_w = widgets.IntText(value=53, description='Starting Head Position:', style=style, layout=layout)
queue_w = widgets.Text(value='98, 183, 37, 122, 14, 124, 65, 67', description='Request Queue (csv):', style=style, layout=layout)
algo_w = widgets.Dropdown(
    options=['FCFS', 'SSTF', 'SCAN', 'C-SCAN', 'LOOK', 'C-LOOK'],
    value='FCFS',
    description='Select Algorithm:',
    style=style,
    layout=layout
)

btn_run = widgets.Button(
    description='Generate Animation 🎬',
    button_style='primary',
    layout=widgets.Layout(width='400px', margin='10px 0px 0px 165px')
)

output_container = widgets.Output()

# --- 4. ANIMATION GENERATION LOGIC ---
def on_click_run(b):
    with output_container:
        clear_output(wait=True)
        print("Calculating and rendering animation... please wait a moment...")
        
        try:
            max_cyl = int(disk_size_w.value)
            head = int(start_head_w.value)
            reqs = [int(x.strip()) for x in queue_w.value.split(",") if x.strip().isdigit()]
            algo = algo_w.value
        except Exception as e:
            print(f"Error parsing inputs: {e}")
            return

        if algo == "FCFS": path, dist = DiskAlgorithms.fcfs(reqs, head)
        elif algo == "SSTF": path, dist = DiskAlgorithms.sstf(reqs, head)
        elif algo == "SCAN": path, dist = DiskAlgorithms.scan(reqs, head, max_cyl)
        elif algo == "C-SCAN": path, dist = DiskAlgorithms.cscan(reqs, head, max_cyl)
        elif algo == "LOOK": path, dist = DiskAlgorithms.look(reqs, head)
        elif algo == "C-LOOK": path, dist = DiskAlgorithms.clook(reqs, head)
 
        colors = {"FCFS": "tab:blue", "SSTF": "tab:green", "LOOK": "tab:orange", 
                  "C-LOOK": "tab:purple", "SCAN": "tab:red", "C-SCAN": "tab:pink"}

        fig, ax = plt.subplots(figsize=(7, 4.5))
        ax.set_xlim(0, max_cyl)
        ax.set_ylim(len(path) - 0.5, -0.5)
        ax.xaxis.tick_top()
        ax.xaxis.set_label_position('top')
        ax.set_xlabel("Cylinders")
        ax.set_ylabel("Time Step")
        ax.set_title(f"{algo} Algorithm (Total Head Movement: {dist})\n", fontweight='bold')
        
        line, = ax.plot([], [], color=colors[algo], linewidth=2)
        scatter = ax.scatter([], [], color=colors[algo], edgecolors='black', zorder=3)

        def update(frame):
            x = path[:frame+1]
            y = list(range(frame+1))
            line.set_data(x, y)
            scatter.set_offsets(np.c_[x, y])
            return line, scatter

        anim = FuncAnimation(fig, update, frames=len(path), interval=500, blit=True)
        plt.close(fig) 
        
        clear_output(wait=True)
        display(HTML(anim.to_html5_video()))

btn_run.on_click(on_click_run)

# --- 5. DISPLAY INTERFACE ---
print("⚙️ DISK SCHEDULER CONFIGURATION PANEL")
display(widgets.VBox([disk_size_w, start_head_w, queue_w, algo_w, btn_run]))
display(output_container)